# Metabolomics VIP analyses

Date last updated: 1/10/26

Notebook author: Yang Chen, Britta De Pessemier

Data analysis by: Britta De Pessemier, Yang Chen

This notebook plots the following:
- Top VIPs separating between pairwise skin groups and univariate statistical testing

In [384]:
# Import Python packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu
import matplotlib.ticker as ticker
from matplotlib.patches import Rectangle
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import Normalize
from brokenaxes import brokenaxes
from matplotlib.colors import LinearSegmentedColormap
import biom
from statsmodels.stats.multitest import multipletests



In [385]:
VIP_LvsH_merged = pd.read_csv('../Data/metabolomics/Run3_10252024/output/Top_Features_VIP_LvsH_merged_data_method2_11212024.csv')
VIP_LvsH_merged['group'] = "H vs L"

VIP_NLvsH_merged = pd.read_csv('../Data/metabolomics/Run3_10252024/output/Top_Features_VIP_NLvsH_merged_data_method2_11212024.csv')
VIP_NLvsH_merged['group'] = "H vs NL"

VIP_NLvsL_merged = pd.read_csv('../Data/metabolomics/Run3_10252024/output/Top_Features_VIP_NLvsL_merged_data_method2_11212024.csv')
VIP_NLvsL_merged['group'] = "NL vs L"

VIP_LvsH = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIPs_LvsH_Load_data_method2_11212024.csv')
VIP_LvsH['group'] = "L vs H"

VIP_NLvsH = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIPs_NLvsH_Load_data_method2_11212024.csv')
VIP_NLvsH['group'] = "NL vs H"

VIP_NLvsL = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIPs_NLvsL_Load_data_method2_11212024.csv')
VIP_NLvsL['group'] = "NL vs L"

# Replace values in the GroupContrib column
VIP_NLvsL_merged['GroupContrib'] = VIP_NLvsL_merged['GroupContrib'].replace({
    'Acne Lesional': 'Acne_L',
    'Acne Non-lesional': 'Acne_NL'
})
# Replace values in the GroupContrib column
VIP_NLvsL['GroupContrib'] = VIP_NLvsL['GroupContrib'].replace({
    'Acne Lesional': 'Acne_L',
    'Acne Non-lesional': 'Acne_NL'
})

VIP_NLvsH_merged['GroupContrib'] = VIP_NLvsH_merged['GroupContrib'].replace({
    'Acne Non-lesional': 'Acne_NL',
    'Healthy': 'Healthy'
})

VIP_NLvsH['GroupContrib'] = VIP_NLvsH['GroupContrib'].replace({
    'Acne Non-lesional': 'Acne_NL',
    'Healthy': 'Healthy'
})

VIP_LvsH_merged['GroupContrib'] = VIP_LvsH_merged['GroupContrib'].replace({
    'Acne Lesional': 'Acne_L',
    'Healthy': 'Healthy'
})

VIP_LvsH['GroupContrib'] = VIP_LvsH['GroupContrib'].replace({
    'Acne Lesional': 'Acne_L',
    'Healthy': 'Healthy'
})


In [386]:
# Concatenate the DataFrames vertically
VIP_combined = pd.concat([VIP_LvsH_merged, VIP_NLvsH_merged, VIP_NLvsL_merged], ignore_index=True)
#VIP_combined_Loadings = pd.concat([VIP_LvsH, VIP_NLvsH, VIP_NLvsL], ignore_index=True)

# Sort the combined DataFrame by the 'VIP' column in descending order
VIP_sorted = VIP_combined.sort_values(by='comp1', ascending=False)

# Replace NaN values in 'superclass', 'superclass', and 'superclass' columns with "Unknown"
VIP_sorted[['superclass', 'class', 'subclass']] = VIP_sorted[['superclass', 'class', 'subclass']].fillna("Unknown")

#VIP_sorted.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_all.csv')

# Filter out rows where VIP < 1
VIP_filtered = VIP_sorted[VIP_sorted['comp1'] >= 1]

# Filter out rows where 'molecular_formula' is NaN
VIP_filtered = VIP_filtered[VIP_filtered['molecular_formula'].notna()]


In [387]:
# Define a function to calculate the VIP_direction
def calculate_vip_direction(row):
    if row['group'] == 'H vs L':
        return row['comp1'] if row['GroupContrib'] == 'Acne_L' else -row['comp1']
    elif row['group'] == 'H vs NL':
        return row['comp1'] if row['GroupContrib'] == 'Acne_NL' else -row['comp1']
    elif row['group'] == 'NL vs L':
        return row['comp1'] if row['GroupContrib'] == 'Acne_L' else -row['comp1']
    else:
        return row['comp1']  # Default to keeping the value unchanged if no match

# Apply the function to create the new column
VIP_filtered['VIP_direction'] = VIP_filtered.apply(calculate_vip_direction, axis=1)

VIP_filtered.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_data_method2_11212024.csv')
#VIP_filtered.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final.csv')

In [388]:
# VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_name.csv')
VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_data_method2_11212024.csv')

# Filter out rows where GroupContrib is NaN
VIP_filtered = VIP_filtered.dropna(subset=['GroupContrib'])
VIP_filtered.head()

,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,...,InChIKey.Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,VIP_direction
0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,...,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219
1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,...,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565
2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,...,QIVBCDIJIAJPQS,Organoheterocyclic compounds,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726
3,20,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,3738000,...,SNICXCGAKADSCV,Organoheterocyclic compounds,Pyridines and derivatives,Pyrrolidinylpyridines,Nicotinic acid alkaloids,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL,-2.116338
4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,...,NWGKJDSIEKMTRX,Lipids and lipid-like molecules,Fatty Acyls,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722


In [389]:
# read in info_feature_complete.csv
info_feature_complete = pd.read_csv('../Data/metabolomics/Run3_10252024/info_feature_complete.csv')
info_feature_complete

,Feature,mz,RT,Corr_ID,Compound_Name,Adduct,formulaRank,molecularFormula,adduct,precursorFormula,...,ClassyFire#level 5 Probability,ClassyFire#most specific class,ClassyFire#most specific class Probability,ClassyFire#all classifications,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,overallFeatureQuality
0,39,100.011014,0.532488,322.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4031,100.075555,1.185771,NaN,NaN,NaN,1.0,C5H9NO,[M + H]+,C5H10NO+,...,0.956,Carboxylic acid amides,0.956,Organic compounds; Organic acids and derivativ...,100.076,76.0,1.259,6.384515e+17,6.377726e+17,NaN
2,14841,100.075569,4.149761,315.0,NaN,NaN,1.0,C5H9NO,[M + H]+,C5H10NO+,...,NaN,Fatty amides,0.597,Organic compounds; Lipids and lipid-like molec...,100.076,249.0,4.146,6.384625e+17,6.377727e+17,NaN
3,32751,101.070841,8.205414,315.0,NaN,NaN,1.0,C4H8N2O,[M + H]+,C4H9N2O+,...,0.576,Acetamides,0.654,Organic compounds; Organic acids and derivativ...,101.071,488.0,8.139,6.384757e+17,6.377728e+17,NaN
4,29425,101.070861,7.750916,315.0,NaN,NaN,1.0,C4H8N2O,[M + H]+,C4H9N2O+,...,0.519,Acetamides,0.515,Organic compounds; Organic acids and derivativ...,101.071,468.0,7.796,6.384755e+17,6.377728e+17,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7678,11341,1122.473419,3.487189,315.0,NaN,NaN,4.0,C62H67N5O15,[M + H]+,C62H68N5O15+,...,0.920,N-acyl-alpha amino acids and derivatives,0.655,Organic compounds; Organoheterocyclic compound...,1122.474,209.0,3.490,6.384586e+17,6.377727e+17,NaN
7679,32902,1124.819915,8.394709,315.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7680,30499,1140.815436,7.715343,315.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7681,28336,1158.805147,7.275935,315.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [390]:
# Merge mz info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'mz']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

# Optionally drop the redundant 'Feature' column
VIP_filtered = VIP_filtered.drop(columns=['Feature'])

In [391]:
# Merge RT info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'RT']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

# Optionally drop the redundant 'Feature' column
VIP_filtered = VIP_filtered.drop(columns=['Feature'])

In [392]:
# Merge ClassyFire info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'ClassyFire#most specific class']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

# Optionally drop the redundant 'Feature' column
VIP_filtered = VIP_filtered.drop(columns=['Feature'])
# VIP_filtered.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_with_shortened_names.csv')
VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_with_shortened_names.tsv', sep='\t')
VIP_filtered

,Unnamed: 0.1,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,...,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,VIP_direction,mz,RT,ClassyFire#most specific class
0,0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219,311.257397,7.421430,Fatty alcohols
1,1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565,311.257397,7.421430,Fatty alcohols
2,2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726,205.096918,2.076001,Alpha amino acids
3,3,20,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,...,Pyrrolidinylpyridines,Nicotinic acid alkaloids,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL,-2.116338,163.122416,0.634317,Pyrrolidinylpyridines
4,4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,...,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722,428.372821,6.156398,Alpha amino acids and derivatives
5,5,23,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL,1.922795,205.096918,2.076001,Alpha amino acids
6,6,49,3794,1.824152,Acne_NL,-0.054289,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,...,"Amino acids, peptides, and analogues",Small peptides,Dipeptides,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005747606,NL vs L,-1.824152,311.123857,1.203634,Gamma-glutamyl amino acids
7,7,50,17958,1.823974,Acne_L,0.054284,CCMSLIB00012283450,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.776449,...,O-methylated flavonoids,Flavonoids,Flavones,Shikimates and Phenylpropanoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012283450,NL vs L,1.823974,373.127569,5.024735,5-O-methylated flavonoids
8,8,24,27758,1.814812,Acne_NL,-0.054131,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,...,Amines,Fatty amides,N-acyl ethanolamines (endocannabinoids),Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003137348,H vs NL,1.814812,326.305056,7.132356,N-acyl amines
9,9,52,777,1.758563,Acne_L,0.052337,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,...,Pyrrolidinylpyridines,Nicotinic acid alkaloids,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,NL vs L,1.758563,163.122416,0.634317,Pyrrolidinylpyridines


In [393]:
# Merge ClassyFire info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'ClassyFire#subclass']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

# Optionally drop the redundant 'Feature' column
VIP_filtered = VIP_filtered.drop(columns=['Feature'])
VIP_filtered

,Unnamed: 0.1,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,...,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,VIP_direction,mz,RT,ClassyFire#most specific class,ClassyFire#subclass
0,0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219,311.257397,7.421430,Fatty alcohols,Fatty alcohols
1,1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565,311.257397,7.421430,Fatty alcohols,Fatty alcohols
2,2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726,205.096918,2.076001,Alpha amino acids,"Amino acids, peptides, and analogues"
3,3,20,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,...,Nicotinic acid alkaloids,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL,-2.116338,163.122416,0.634317,Pyrrolidinylpyridines,Pyrrolidinylpyridines
4,4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,...,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722,428.372821,6.156398,Alpha amino acids and derivatives,"Amino acids, peptides, and analogues"
5,5,23,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL,1.922795,205.096918,2.076001,Alpha amino acids,"Amino acids, peptides, and analogues"
6,6,49,3794,1.824152,Acne_NL,-0.054289,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,...,Small peptides,Dipeptides,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005747606,NL vs L,-1.824152,311.123857,1.203634,Gamma-glutamyl amino acids,"Amino acids, peptides, and analogues"
7,7,50,17958,1.823974,Acne_L,0.054284,CCMSLIB00012283450,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.776449,...,Flavonoids,Flavones,Shikimates and Phenylpropanoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012283450,NL vs L,1.823974,373.127569,5.024735,5-O-methylated flavonoids,O-methylated flavonoids
8,8,24,27758,1.814812,Acne_NL,-0.054131,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,...,Fatty amides,N-acyl ethanolamines (endocannabinoids),Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003137348,H vs NL,1.814812,326.305056,7.132356,N-acyl amines,Fatty amides
9,9,52,777,1.758563,Acne_L,0.052337,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,...,Nicotinic acid alkaloids,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,NL vs L,1.758563,163.122416,0.634317,Pyrrolidinylpyridines,Pyrrolidinylpyridines


In [394]:
# Merge ClassyFire info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'ClassyFire#class']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

# Optionally drop the redundant 'Feature' column
VIP_filtered = VIP_filtered.drop(columns=['Feature'])
VIP_filtered

,Unnamed: 0.1,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,...,npclassifier_class,npclassifier_pathway,library_usi,group,VIP_direction,mz,RT,ClassyFire#most specific class,ClassyFire#subclass,ClassyFire#class
0,0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219,311.257397,7.421430,Fatty alcohols,Fatty alcohols,Fatty Acyls
1,1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565,311.257397,7.421430,Fatty alcohols,Fatty alcohols,Fatty Acyls
2,2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726,205.096918,2.076001,Alpha amino acids,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives
3,3,20,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,...,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL,-2.116338,163.122416,0.634317,Pyrrolidinylpyridines,Pyrrolidinylpyridines,Pyridines and derivatives
4,4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,...,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722,428.372821,6.156398,Alpha amino acids and derivatives,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives
5,5,23,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL,1.922795,205.096918,2.076001,Alpha amino acids,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives
6,6,49,3794,1.824152,Acne_NL,-0.054289,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,...,Dipeptides,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005747606,NL vs L,-1.824152,311.123857,1.203634,Gamma-glutamyl amino acids,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives
7,7,50,17958,1.823974,Acne_L,0.054284,CCMSLIB00012283450,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.776449,...,Flavones,Shikimates and Phenylpropanoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012283450,NL vs L,1.823974,373.127569,5.024735,5-O-methylated flavonoids,O-methylated flavonoids,Flavonoids
8,8,24,27758,1.814812,Acne_NL,-0.054131,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,...,N-acyl ethanolamines (endocannabinoids),Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003137348,H vs NL,1.814812,326.305056,7.132356,N-acyl amines,Fatty amides,Fatty Acyls
9,9,52,777,1.758563,Acne_L,0.052337,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,...,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,NL vs L,1.758563,163.122416,0.634317,Pyrrolidinylpyridines,Pyrrolidinylpyridines,Pyridines and derivatives


In [395]:
# Merge ClassyFire info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'NPC#superclass']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

# Optionally drop the redundant 'Feature' column
VIP_filtered = VIP_filtered.drop(columns=['Feature'])
VIP_filtered.head()

,Unnamed: 0.1,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,...,npclassifier_pathway,library_usi,group,VIP_direction,mz,RT,ClassyFire#most specific class,ClassyFire#subclass,ClassyFire#class,NPC#superclass
0,0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219,311.257397,7.421430,Fatty alcohols,Fatty alcohols,Fatty Acyls,Fatty Acids and Conjugates
1,1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565,311.257397,7.421430,Fatty alcohols,Fatty alcohols,Fatty Acyls,Fatty Acids and Conjugates
2,2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726,205.096918,2.076001,Alpha amino acids,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives,Small peptides
3,3,20,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,...,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL,-2.116338,163.122416,0.634317,Pyrrolidinylpyridines,Pyrrolidinylpyridines,Pyridines and derivatives,Nicotinic acid alkaloids
4,4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,...,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722,428.372821,6.156398,Alpha amino acids and derivatives,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives,Spingolipids


In [396]:
# Merge ClassyFire info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'ClassyFire#superclass']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

# Optionally drop the redundant 'Feature' column
VIP_filtered = VIP_filtered.drop(columns=['Feature'])
VIP_filtered.head()

,Unnamed: 0.1,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,...,library_usi,group,VIP_direction,mz,RT,ClassyFire#most specific class,ClassyFire#subclass,ClassyFire#class,NPC#superclass,ClassyFire#superclass
0,0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219,311.257397,7.421430,Fatty alcohols,Fatty alcohols,Fatty Acyls,Fatty Acids and Conjugates,Lipids and lipid-like molecules
1,1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565,311.257397,7.421430,Fatty alcohols,Fatty alcohols,Fatty Acyls,Fatty Acids and Conjugates,Lipids and lipid-like molecules
2,2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726,205.096918,2.076001,Alpha amino acids,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives,Small peptides,Organic acids and derivatives
3,3,20,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL,-2.116338,163.122416,0.634317,Pyrrolidinylpyridines,Pyrrolidinylpyridines,Pyridines and derivatives,Nicotinic acid alkaloids,Organoheterocyclic compounds
4,4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722,428.372821,6.156398,Alpha amino acids and derivatives,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives,Spingolipids,Organic acids and derivatives


### Perform univariate statistical testing on above metabolites w/ top VIPs

In [397]:
# Function to load BIOM table, collapse by taxa, sort rows by row sum
def load_biom_table(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)
    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    df = df.T
    
    return df

In [398]:
# Read in metabolomics table
#metaB_table = pd.read_csv('../Data/metabolomics/Run3_10252024/data_sample_10282024.csv')

# metaB_table = pd.read_csv('../Data/metabolomics/Run3_10252024/metabolomics_data_02072025_sample_name.csv')
metaB_method2 = load_biom_table('../Data/metabolomics/Run3_10252024/metabolomics_method2.biom')
metaB_method2

,4293,15974,2531,5907,5946,2965,22668,1032,1074,955,...,25423,30708,28077,25841,30835,25998,33060,30635,26562,28336
LAMI.RD001.D0.C1,4770.76270,0.0000,72812.695,1022406.25,0.0,383280.300,35353.773,44675.1600,9472625.00,55584.406,...,133534.610,0.00,2788.4210,0.000,0.000,98869.660,0.0000,0.0000,44668.117,8376.1850
LAMI.RD308.D2.C2,4166.87100,0.0000,56668.180,322180.03,0.0,278639.000,70091.570,0.0000,896991.75,2362.046,...,265390.100,86837.78,20559.6040,65032.520,46506.953,366275.560,0.0000,0.0000,110988.580,11684.1080
LAMI.RD304.D11.C1,2342.99900,0.0000,116903.530,548906.90,0.0,232978.390,36155.680,47782.8440,11899350.00,51932.010,...,205414.810,0.00,0.0000,84496.870,0.000,106990.060,0.0000,0.0000,66076.950,8903.7910
LAMI.RD304.D11.C2,740.01514,6008.3945,69846.520,303948.28,0.0,114657.980,52892.070,24622.6290,5926156.50,25971.598,...,158658.450,0.00,0.0000,25730.426,0.000,55033.453,1521.3207,0.0000,19372.977,8350.3290
LAMI.RD304.D0.C1,725.83435,6228.5117,75098.500,438357.53,0.0,106253.080,45601.550,6258.2560,3217912.80,18388.262,...,125322.016,0.00,0.0000,28067.354,0.000,107743.010,0.0000,0.0000,49133.240,10825.2950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.00000,0.0000,53799.130,458503.40,0.0,219629.830,22518.434,12877.7910,3450308.50,16107.057,...,13180.887,0.00,12656.4730,0.000,0.000,40761.030,0.0000,0.0000,15749.498,5134.8340
LAMI.RD308.D9.C3,0.00000,0.0000,30274.117,1192993.40,0.0,98506.914,0.000,0.0000,398077.20,0.000,...,97591.470,71785.20,1711.5046,0.000,39292.410,14812.991,0.0000,0.0000,0.000,0.0000
LAMI.RD319.D2.C2,1567.89330,0.0000,63744.797,458884.62,0.0,92721.800,25671.117,3202.1710,1813529.90,4184.649,...,123806.360,0.00,0.0000,0.000,0.000,100497.234,0.0000,0.0000,0.000,2682.9302
LAMI.RD319.D2.C3,0.00000,25945.3710,47938.910,403263.28,0.0,19834.994,22296.582,0.0000,662763.30,0.000,...,67352.125,31273.33,8610.4320,0.000,0.000,13816.301,9280.9160,1639.8352,0.000,5966.7130


In [399]:
# Set the 'SampleID' column as the index
# metaB_table.set_index('SampleID', inplace=True)

# Convert columns in metaB_table to int (if applicable)
metaB_table.columns = metaB_table.columns.astype(int)

# Convert features_to_keep to int
features_to_keep = VIP_filtered['ID'].unique().astype(int)

# Subset metaB_table using the integer feature list
metaB_table_subset = metaB_table.loc[:, metaB_table.columns.isin(features_to_keep)]

metaB_table_subset

,954,1368,941,655,5062,777,2969,1119,5930,3496,...,1197,3794,29059,27758,23297,17958,19710,24392,20967,11925
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.0,970487.500,1942604.500,567510.060,1473068.10,...,62007.1100,12406.8260,611709.700,16778.8380,0.0000,0.000,0.000,0.00,0.0000,125692.3300
LAMI.RD308.D2.C2,1121916.40,1669351.50,2905.5796,3065.1714,19280.236,0.0,750870.100,1618142.800,442653.660,286004.00,...,87817.2000,5348.9400,231525.840,14299.3520,0.0000,0.000,0.000,0.00,3822.6377,4734.5693
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.0,595568.940,1605551.000,328151.120,299140.06,...,58588.9260,9403.6520,78491.530,0.0000,5375.8525,11225.194,23546.951,99470.69,29824.6270,12473.0920
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.0,303845.300,865753.500,152392.720,191713.67,...,20153.3320,2286.2397,261996.800,15780.3090,2691.0256,16378.752,34863.656,157503.92,44084.5860,40673.1100
LAMI.RD304.D0.C1,827466.30,778566.10,4547.2640,3345.8184,45187.030,0.0,293386.340,1321474.400,137201.390,705570.80,...,45193.9400,0.0000,427202.060,3228.1143,3431.0034,0.000,0.000,0.00,0.0000,49464.9020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.0,579058.800,1380958.500,498893.780,236111.75,...,24563.9240,5144.6616,122958.260,56654.0430,0.0000,0.000,0.000,0.00,0.0000,0.0000
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.0,281896.700,742416.600,262886.470,303509.40,...,24480.3900,0.0000,65659.700,21826.1170,0.0000,0.000,0.000,0.00,0.0000,220846.1600
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.0,255542.720,1194061.900,121803.890,5406879.00,...,35224.5800,0.0000,73231.560,19741.2460,0.0000,0.000,0.000,0.00,0.0000,25695.0140


In [400]:
# Read in metadata
metadata = pd.read_csv('../Metadata/metadata_final_22102024.tsv', sep='\t')

# Subset metadata based on the indices of metaB_subset
metadata_subset = metadata[metadata['SampleID'].isin(metaB_table_subset.index)]

metadata_subset

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group,subject_ID_x_group
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L,moderate,moderate Acne_L,PP_310_Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL,absent,absent Acne_NL,PP_305_Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,low,low Acne_L,PP_306_Acne_L
6,LAMI.RD317.D14.C2,C2,70,Lesional,skin,Day 14,44,14,317,15,...,acne,RD317,acne,PP_317,PP_317C2,Lesional_C2,Acne_L,low,low Acne_L,PP_317_Acne_L
7,LAMI.RD305.D23.C1,C1,not applicable,Lesional,skin,Day 23,not applicable,23,305,not applicable,...,acne,RD305,acne,PP_305,PP_305C1,Lesional_C1,Acne_L,low,low Acne_L,PP_305_Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260,LAMI.RD305.D0.C2,C2,69,Lesional,skin,Day 0,12,0,305,27,...,acne,RD305,acne,PP_305,PP_305C2,Lesional_C2,Acne_L,moderate,moderate Acne_L,PP_305_Acne_L
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L,low,low Acne_L,PP_317_Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_14_Healthy


In [401]:
# Map the 'group' column from metadata_subset to metaB_subset
metaB_table_subset['group'] = metaB_table_subset.index.map(metadata_subset.set_index('SampleID')['group'])

# Replace values in the 'group' column
metaB_table_subset['group'] = metaB_table_subset['group'].replace({
    'Healthy': 'H',
    'Acne_L': 'L',
    'Acne_NL': 'NL'
})

metaB_table_subset

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/2462488200.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metaB_table_subset['group'] = metaB_table_subset.index.map(metadata_subset.set_index('SampleID')['group'])
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/2462488200.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metaB_table_subset['group'] = metaB_table_subset['group'].replace({


,954,1368,941,655,5062,777,2969,1119,5930,3496,...,3794,29059,27758,23297,17958,19710,24392,20967,11925,group
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.0,970487.500,1942604.500,567510.060,1473068.10,...,12406.8260,611709.700,16778.8380,0.0000,0.000,0.000,0.00,0.0000,125692.3300,H
LAMI.RD308.D2.C2,1121916.40,1669351.50,2905.5796,3065.1714,19280.236,0.0,750870.100,1618142.800,442653.660,286004.00,...,5348.9400,231525.840,14299.3520,0.0000,0.000,0.000,0.00,3822.6377,4734.5693,L
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.0,595568.940,1605551.000,328151.120,299140.06,...,9403.6520,78491.530,0.0000,5375.8525,11225.194,23546.951,99470.69,29824.6270,12473.0920,L
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.0,303845.300,865753.500,152392.720,191713.67,...,2286.2397,261996.800,15780.3090,2691.0256,16378.752,34863.656,157503.92,44084.5860,40673.1100,L
LAMI.RD304.D0.C1,827466.30,778566.10,4547.2640,3345.8184,45187.030,0.0,293386.340,1321474.400,137201.390,705570.80,...,0.0000,427202.060,3228.1143,3431.0034,0.000,0.000,0.00,0.0000,49464.9020,L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.0,579058.800,1380958.500,498893.780,236111.75,...,5144.6616,122958.260,56654.0430,0.0000,0.000,0.000,0.00,0.0000,0.0000,NL
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.0,281896.700,742416.600,262886.470,303509.40,...,0.0000,65659.700,21826.1170,0.0000,0.000,0.000,0.00,0.0000,220846.1600,NL
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.0,255542.720,1194061.900,121803.890,5406879.00,...,0.0000,73231.560,19741.2460,0.0000,0.000,0.000,0.00,0.0000,25695.0140,L


In [402]:
# Prepare the dictionary to set up univariate testing
feature_group_dict = (
    VIP_filtered.groupby('ID')['group']
    .apply(lambda x: x.unique().tolist())
    .to_dict()
)

In [403]:
# Initialize an empty dictionary to store results
statistical_results = {}

# Iterate through each metabolite in feature_group_dict
for metabolite, comparisons in feature_group_dict.items():
    # Create a dictionary to store results for the current metabolite
    metabolite_results = {}
    
    # Get the values for the metabolite from metaB_table_subset
    metabolite_values = metaB_table_subset[metabolite]
    
    # Iterate through each comparison for the current metabolite
    for comparison in comparisons:
        # Split the comparison into X and Y
        group_x, group_y = comparison.split(" vs ")
        
        # Subset values for the two groups
        values_x = metabolite_values[metaB_table_subset['group'] == group_x]
        values_y = metabolite_values[metaB_table_subset['group'] == group_y]
        
        # Perform the Mann-Whitney U test
        stat, p_value = mannwhitneyu(values_x, values_y, alternative='two-sided')
        
        # Store the result
        metabolite_results[comparison] = {'U-statistic': stat, 'p-value': p_value}
    
    # Add results for the current metabolite to the main results dictionary
    statistical_results[metabolite] = metabolite_results

# Convert results to a DataFrame for easier viewing (optional)
results_df = pd.DataFrame.from_dict({(met, comp): vals 
                                     for met, comps in statistical_results.items() 
                                     for comp, vals in comps.items()}, 
                                    orient='index')


# View results
results_df


U-statistic   p-value
379   NL vs L       2506.0  0.118156
      H vs NL        575.0  0.246680
655   NL vs L       2622.0  0.035492
      H vs NL        606.0  0.405200
777   H vs NL        876.5  0.027488
      NL vs L       2071.0  0.767129
      H vs L        2942.0  0.016249
862   NL vs L       2472.0  0.155851
      H vs NL        590.0  0.319225
941   H vs L        2218.0  0.526262
954   H vs NL        551.0  0.156106
      H vs L        2237.0  0.576631
1119  NL vs L       2486.0  0.139352
      H vs NL        639.0  0.641541
1197  NL vs L       2495.5  0.128945
      H vs NL        598.0  0.363119
1368  H vs NL        538.0  0.118988
      NL vs L       2558.0  0.074771
      H vs L        2222.0  0.536907
2969  H vs NL        531.0  0.102086
      NL vs L       2579.0  0.061424
      H vs L        2189.0  0.454638
3496  H vs NL        805.0  0.186551
3794  NL vs L       2513.0  0.109209
      H vs L        2275.0  0.680104
5062  H vs NL        626.0  0.544321
      H vs L        2367.0  0.963808
5930  NL vs L       2769.0  0.007546
      H vs NL        501.0  0.050040
      H vs L        2193.0  0.464216
7778  H vs NL        644.0  0.584706
      H vs L        2513.0  0.431753
11925 H vs NL        610.0  0.426629
      H vs L        2264.0  0.644373
15770 NL vs L       2142.0  0.963042
17958 NL vs L       2156.0  0.881355
19710 NL vs L       2136.0  0.997156
20967 H vs NL        625.0  0.224219
      NL vs L       2142.0  0.963042
23297 H vs L        1725.5  0.000763
      NL vs L       2404.5  0.177976
24392 NL vs L       1743.0  0.080255
      H vs NL        669.0  0.872994
27758 H vs NL        558.0  0.179420
      H vs L        2232.5  0.564559
29059 H vs NL        469.0  0.021106
      H vs L        1864.0  0.042352

In [404]:
VIP_filtered['Compound_Name']

0     NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...
1     NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...
2                                D-TRYPTOPHAN - 60.0 eV
3     Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...
4     Sorbitane Monooleate - Polysorbate 20 in-sourc...
5                                D-TRYPTOPHAN - 60.0 eV
6                    Massbank:PR311057 Glutamyltyrosine
7                                            Sinensetin
8     Spectral Match to N-Oleoylethanolamine from NI...
9     Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...
10                    Phenylbenzimidazole sulfonic acid
11                   Massbank:PR311057 Glutamyltyrosine
12                             Phenylalanine - 50.00 eV
13                                 ISOLEUCINE - 60.0 eV
14                                        UROCANIC ACID
15                                       xylometazoline
16    Spectral Match to N-Oleoylethanolamine from NI...
17                                              

In [405]:
# Ensure the Feature column in VIP_filtered has unique mappings
VIP_filtered_unique = VIP_filtered.drop_duplicates(subset='ID')

# Check if results_df has a MultiIndex
if isinstance(results_df.index, pd.MultiIndex):
    # Use the first level of the MultiIndex for mapping
    feature_index = results_df.index.get_level_values(0).astype(int)
else:
    # Use the index directly if not a MultiIndex
    feature_index = results_df.index.astype(int)

# Ensure Feature column in VIP_filtered is of the same type
VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)

# Create a mapping from Feature to Shortened_Compound_Name
feature_to_name_mapping = VIP_filtered_unique.set_index('ID')['Shortened_Compound_Name']
feature_to_name_mapping = VIP_filtered_unique.set_index('ID')['Compound_Name']

# Map the Shortened_Compound_Name to results_df
results_df['Shortened_Compound_Name'] = feature_index.map(feature_to_name_mapping)
results_df['Compound_Name'] = feature_index.map(feature_to_name_mapping)
results_df


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/746498369.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)


U-statistic   p-value  \
379   NL vs L       2506.0  0.118156   
      H vs NL        575.0  0.246680   
655   NL vs L       2622.0  0.035492   
      H vs NL        606.0  0.405200   
777   H vs NL        876.5  0.027488   
      NL vs L       2071.0  0.767129   
      H vs L        2942.0  0.016249   
862   NL vs L       2472.0  0.155851   
      H vs NL        590.0  0.319225   
941   H vs L        2218.0  0.526262   
954   H vs NL        551.0  0.156106   
      H vs L        2237.0  0.576631   
1119  NL vs L       2486.0  0.139352   
      H vs NL        639.0  0.641541   
1197  NL vs L       2495.5  0.128945   
      H vs NL        598.0  0.363119   
1368  H vs NL        538.0  0.118988   
      NL vs L       2558.0  0.074771   
      H vs L        2222.0  0.536907   
2969  H vs NL        531.0  0.102086   
      NL vs L       2579.0  0.061424   
      H vs L        2189.0  0.454638   
3496  H vs NL        805.0  0.186551   
3794  NL vs L       2513.0  0.109209   
      H vs L        2275.0  0.680104   
5062  H vs NL        626.0  0.544321   
      H vs L        2367.0  0.963808   
5930  NL vs L       2769.0  0.007546   
      H vs NL        501.0  0.050040   
      H vs L        2193.0  0.464216   
7778  H vs NL        644.0  0.584706   
      H vs L        2513.0  0.431753   
11925 H vs NL        610.0  0.426629   
      H vs L        2264.0  0.644373   
15770 NL vs L       2142.0  0.963042   
17958 NL vs L       2156.0  0.881355   
19710 NL vs L       2136.0  0.997156   
20967 H vs NL        625.0  0.224219   
      NL vs L       2142.0  0.963042   
23297 H vs L        1725.5  0.000763   
      NL vs L       2404.5  0.177976   
24392 NL vs L       1743.0  0.080255   
      H vs NL        669.0  0.872994   
27758 H vs NL        558.0  0.179420   
      H vs L        2232.5  0.564559   
29059 H vs NL        469.0  0.021106   
      H vs L        1864.0  0.042352   

                                         Shortened_Compound_Name  \
379   NL vs L                                            URIDINE   
      H vs NL                                            URIDINE   
655   NL vs L                                      UROCANIC ACID   
      H vs NL                                      UROCANIC ACID   
777   H vs NL  Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
      NL vs L  Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
      H vs L   Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
862   NL vs L                                     Cyclo(his-pro)   
      H vs NL                                     Cyclo(his-pro)   
941   H vs L                                       UROCANIC ACID   
954   H vs NL                      L-Pyroglutamic acid - 40.0 eV   
      H vs L                       L-Pyroglutamic acid - 40.0 eV   
1119  NL vs L           Spectral Match to L-Tyrosine from NIST14   
      H vs NL           Spectral Match to L-Tyrosine from NIST14   
1197  NL vs L           Massbank:PR310825 N-Fructosyl isoleucine   
      H vs NL           Massbank:PR310825 N-Fructosyl isoleucine   
1368  H vs NL                               ISOLEUCINE - 60.0 eV   
      NL vs L                               ISOLEUCINE - 60.0 eV   
      H vs L                                ISOLEUCINE - 60.0 eV   
2969  H vs NL                           Phenylalanine - 50.00 eV   
      NL vs L                           Phenylalanine - 50.00 eV   
      H vs L                            Phenylalanine - 50.00 eV   
3496  H vs NL                                       DL-Panthenol   
3794  NL vs L                 Massbank:PR311057 Glutamyltyrosine   
      H vs L                  Massbank:PR311057 Glutamyltyrosine   
5062  H vs NL                              Paracetamol - 40.0 eV   
      H vs L                               Paracetamol - 40.0 eV   
5930  NL vs L                             D-TRYPTOPHAN - 60.0 eV   
      H vs NL                             D-TRYPTOPHAN - 60.0 eV   
      H vs L                              D-TRYPTOPHAN

In [406]:
# Subset to rows where p-value <= 0.05
results_df_sig = results_df[results_df['p-value'] <= 0.05]

# Initialize column
results_df_sig['Final_Name'] = np.nan

# Assign names based on index
results_df_sig.loc[655, 'Final_Name'] = 'Urocanic acid'
results_df_sig.loc[5930, 'Final_Name'] = 'Tryptophan'
results_df_sig.loc[23297, 'Final_Name'] = 'Gln-C14:0'
results_df_sig.loc[29059, 'Final_Name'] = 'C19H36O4 Fatty alcohol'

results_df_sig


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/1607807321.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_df_sig['Final_Name'] = np.nan
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/1607807321.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Urocanic acid' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  results_df_sig.loc[655, 'Final_Name'] = 'Urocanic acid'


U-statistic   p-value  \
655   NL vs L       2622.0  0.035492   
777   H vs NL        876.5  0.027488   
      H vs L        2942.0  0.016249   
5930  NL vs L       2769.0  0.007546   
23297 H vs L        1725.5  0.000763   
29059 H vs NL        469.0  0.021106   
      H vs L        1864.0  0.042352   

                                         Shortened_Compound_Name  \
655   NL vs L                                      UROCANIC ACID   
777   H vs NL  Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
      H vs L   Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
5930  NL vs L                             D-TRYPTOPHAN - 60.0 eV   
23297 H vs L                                           Gln-C14:0   
29059 H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
      H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   

                                                   Compound_Name  \
655   NL vs L                                      UROCANIC ACID   
777   H vs NL  Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
      H vs L   Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
5930  NL vs L                             D-TRYPTOPHAN - 60.0 eV   
23297 H vs L                                           Gln-C14:0   
29059 H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
      H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   

                           Final_Name  
655   NL vs L           Urocanic acid  
777   H vs NL                     NaN  
      H vs L                      NaN  
5930  NL vs L              Tryptophan  
23297 H vs L                Gln-C14:0  
29059 H vs NL  C19H36O4 Fatty alcohol  
      H vs L   C19H36O4 Fatty alcohol

In [ ]:
# Get list
significant_features = results_df_sig['Final_Name'].dropna().tolist()
significant_features

['Urocanic acid',
 'Tryptophan',
 'Gln-C14:0',
 'C19H36O4 Fatty alcohol',
 'C19H36O4 Fatty alcohol']

In [408]:
# Ensure the Feature column in VIP_filtered has unique mappings
VIP_filtered_unique = VIP_filtered.drop_duplicates(subset='ID')

# Check if results_df has a MultiIndex
if isinstance(results_df.index, pd.MultiIndex):
    # Use the first level of the MultiIndex for mapping
    feature_index = results_df.index.get_level_values(0).astype(int)
else:
    # Use the index directly if not a MultiIndex
    feature_index = results_df.index.astype(int)

# Ensure Feature column in VIP_filtered is of the same type
VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)

# Create a mapping from Feature to VIP values
feature_to_vip_mapping = VIP_filtered_unique.set_index('ID')['comp1']

# Map the VIP values to results_df
results_df['comp1'] = feature_index.map(feature_to_vip_mapping)

# Sort results_df by the 'VIP' column in descending order
results_df = results_df.sort_values(by='comp1', ascending=False)
results_df = results_df.drop((777,), errors='ignore')

results_df.head()

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/1030576968.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/1030576968.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  results_df = results_df.drop((777,), errors='ignore')


U-statistic   p-value  \
29059 H vs L        1864.0  0.042352   
      H vs NL        469.0  0.021106   
5930  H vs L        2193.0  0.464216   
      H vs NL        501.0  0.050040   
      NL vs L       2769.0  0.007546   

                                         Shortened_Compound_Name  \
29059 H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
      H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
5930  H vs L                              D-TRYPTOPHAN - 60.0 eV   
      H vs NL                             D-TRYPTOPHAN - 60.0 eV   
      NL vs L                             D-TRYPTOPHAN - 60.0 eV   

                                                   Compound_Name     comp1  
29059 H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219  
      H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219  
5930  H vs L                              D-TRYPTOPHAN - 60.0 eV  2.271726  
      H vs NL                             D-TRYPTOPHAN - 60.0 eV  2.271726  
      NL vs L                             D-TRYPTOPHAN - 60.0 eV  2.271726

In [409]:
# Filter rows where the p-value is greater than 0.1
# filtered_features = results_df[results_df['p-value'] <= 0.10]
# filtered_features.to_csv('../Data/metabolomics/Run3_10252024/output/metabolomics_significant_features_final.csv')

filtered_features = results_df
filtered_features.head()

U-statistic   p-value  \
29059 H vs L        1864.0  0.042352   
      H vs NL        469.0  0.021106   
5930  H vs L        2193.0  0.464216   
      H vs NL        501.0  0.050040   
      NL vs L       2769.0  0.007546   

                                         Shortened_Compound_Name  \
29059 H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
      H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
5930  H vs L                              D-TRYPTOPHAN - 60.0 eV   
      H vs NL                             D-TRYPTOPHAN - 60.0 eV   
      NL vs L                             D-TRYPTOPHAN - 60.0 eV   

                                                   Compound_Name     comp1  
29059 H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219  
      H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219  
5930  H vs L                              D-TRYPTOPHAN - 60.0 eV  2.271726  
      H vs NL                             D-TRYPTOPHAN - 60.0 eV  2.271726  
      NL vs L                             D-TRYPTOPHAN - 60.0 eV  2.271726

In [410]:
# Group by the primary index and collect the secondary index values
keep_dict = filtered_features.groupby(level=0).apply(lambda df: df.index.get_level_values(1).tolist()).to_dict()

In [411]:
# Function to check if the Feature and group combination is in keep_dict
def filter_rows(row):
    feature = row['ID']
    group = row['group']
    return feature in keep_dict and group in keep_dict[feature]

# Apply the filter function to VIP_filtered
VIP_filtered_subset = VIP_filtered[VIP_filtered.apply(filter_rows, axis=1)]

In [412]:
# Ensure Feature in VIP_filtered_subset and the primary index in filtered_features are the same type
VIP_filtered_subset['ID'] = VIP_filtered_subset['ID'].astype(int)
filtered_features.index = filtered_features.index.set_levels(
    filtered_features.index.levels[0].astype(int), level=0
)

# Create a MultiIndex mapping of (Feature, group) to p-value
p_value_mapping = filtered_features['p-value']

# Create a new column in VIP_filtered_subset for the p_value_bh
VIP_filtered_subset['p-value'] = VIP_filtered_subset.set_index(['ID', 'group']).index.map(p_value_mapping)

# Reset the index to return to the original structure if needed
VIP_filtered_subset.reset_index(drop=True, inplace=True)

# Map the name from Shortened_Compound_Name
# Drop duplicate IDs, keeping the first occurrence
# unique_mapping = filtered_features.drop_duplicates(subset='ID').set_index('ID')['Shortened_Compound_Name']

# Now map the values
# VIP_filtered_subset['Shortened_Compound_Name'] = VIP_filtered_subset['ID'].map(unique_mapping)
# VIP_filtered_subset.loc[VIP_filtered_subset['Shortened_Compound_Name'] == 'Tryptophan', 'subclass'] = 'Amino acids, peptides, and analogues'


# View the updated DataFrame
VIP_filtered_subset


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/4233722397.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_subset['ID'] = VIP_filtered_subset['ID'].astype(int)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/4233722397.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_subset['p-value'] = VIP_filtered_subset.set_index(['ID', 'group']).index.map(p_value_mapping)


,Unnamed: 0.1,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,...,group,VIP_direction,mz,RT,ClassyFire#most specific class,ClassyFire#subclass,ClassyFire#class,NPC#superclass,ClassyFire#superclass,p-value
0,0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,H vs NL,2.594219,311.257397,7.421430,Fatty alcohols,Fatty alcohols,Fatty Acyls,Fatty Acids and Conjugates,Lipids and lipid-like molecules,0.021106
1,1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,...,H vs L,2.358565,311.257397,7.421430,Fatty alcohols,Fatty alcohols,Fatty Acyls,Fatty Acids and Conjugates,Lipids and lipid-like molecules,0.042352
2,2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,NL vs L,-2.271726,205.096918,2.076001,Alpha amino acids,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives,Small peptides,Organic acids and derivatives,0.007546
3,4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,...,NL vs L,1.947722,428.372821,6.156398,Alpha amino acids and derivatives,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives,Spingolipids,Organic acids and derivatives,0.080255
4,5,23,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,...,H vs NL,1.922795,205.096918,2.076001,Alpha amino acids,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives,Small peptides,Organic acids and derivatives,0.050040
5,6,49,3794,1.824152,Acne_NL,-0.054289,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,...,NL vs L,-1.824152,311.123857,1.203634,Gamma-glutamyl amino acids,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives,Small peptides,Organic acids and derivatives,0.109209
6,7,50,17958,1.823974,Acne_L,0.054284,CCMSLIB00012283450,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.776449,...,NL vs L,1.823974,373.127569,5.024735,5-O-methylated flavonoids,O-methylated flavonoids,Flavonoids,Flavonoids,Phenylpropanoids and polyketides,0.881355
7,8,24,27758,1.814812,Acne_NL,-0.054131,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,...,H vs NL,1.814812,326.305056,7.132356,N-acyl amines,Fatty amides,Fatty Acyls,Fatty amides,Lipids and lipid-like molecules,0.179420
8,10,25,7778,1.729160,Acne_NL,-0.051577,CCMSLIB00000579531,spectra_filtered.mgf,CASMI.mgf,0.968451,...,H vs NL,1.729160,275.048184,2.761023,Phenylbenzimidazoles,Phenylbenzimidazoles,Benzimidazoles,Anthranilic acid alkaloids,Organoheterocyclic compounds,0.584706
9,11,1,3794,1.716047,Healthy,-0.051049,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,...,H vs L,-1.716047,311.123857,1.203634,Gamma-glutamyl amino acids,"Amino acids, peptides, and analogues",Carboxylic acids and derivatives,Small peptides,Organic acids and derivatives,0.680104


In [413]:
# Set up the figure
fig, axes = plt.subplots(
    1, 3, figsize=(20, 6),
    gridspec_kw={"wspace": 0.75}
)

unique_groups = ["H vs L", "H vs NL", "NL vs L"]

# Force Tyrosine to be classified as Alpha amino acids
VIP_filtered_subset.loc[
    VIP_filtered_subset['Shortened_Compound_Name'].str.lower() == 'tyrosine',
    'ClassyFire#most specific class'
] = 'Alpha amino acids'


# Get all unique class categories
unique_classes = VIP_filtered_subset['ClassyFire#most specific class'].dropna().unique()
palette = sns.color_palette("hls", len(unique_classes))
color_map = dict(zip(unique_classes, palette))

fig.suptitle('Top VIP Features Across Groups (VIP > 1)', fontsize=20, y=1.025)

for i, group in enumerate(unique_groups):
    group_data = VIP_filtered_subset[VIP_filtered_subset['group'] == group].copy()

    # Sort by abs(VIP score) descending
    group_data = group_data.sort_values('VIP_direction', key=lambda x: x.abs(), ascending=False).reset_index(drop=True)

    vip_scores = group_data['VIP_direction']
    
    # Format m/z_RT for y-axis labels
    # ids = group_data.apply(lambda row: f"{row['mz']:.4f}_{row['RT']:.2f}", axis=1)
    ids = group_data['Shortened_Compound_Name']

    classes = group_data['ClassyFire#most specific class']
    dot_colors = classes.map(color_map).fillna('gray')

    # Plot
    axes[i].scatter(
        vip_scores,
        range(len(group_data)),
        color=dot_colors,
        s=200
    )

    # VIP cutoff lines
    axes[i].axvline(x=1, color='gray', linestyle='--', linewidth=1.5)
    axes[i].axvline(x=-1, color='gray', linestyle='--', linewidth=1.5)

    # Y-axis = m/z_RT labels
    axes[i].set_yticks(range(len(ids)))
    axes[i].set_yticklabels(ids, fontsize=10)
    axes[i].invert_yaxis()
    axes[i].set_xlabel('VIP Scores', fontsize=12)
    # axes[i].set_ylabel('m/z_RT', fontsize=12, labelpad=10)

    custom_titles = {
        "H vs L": "Healthy vs Lesional",
        "H vs NL": "Healthy vs Non-lesional",
        "NL vs L": "Non-lesional vs Lesional"
    }
    axes[i].set_title(custom_titles.get(group, group), fontsize=14, pad=10)

# Legend for ClassyFire classes
legend_handles = [plt.Line2D([0], [0], marker='o', color=color, linestyle='', markersize=10, label=cls)
                  for cls, color in color_map.items()]

# Add entry for unknown class (gray)
# legend_handles.append(
#     plt.Line2D([0], [0], marker='o', color='gray', linestyle='', markersize=10, label='Unknown')
# )

fig.legend(handles=legend_handles, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.24),
           fontsize=12, title='ClassyFire Most Specific Class', title_fontsize=14, frameon=False)

# Final layout and save
plt.tight_layout(rect=[0, 0.15, 0.9, 1])
plt.savefig("../Figures/Main/Figure_5/VIP_combined_plot_by_mzRT_sorted_colored_most-specific-class.svg", dpi=600, bbox_inches='tight')

plt.savefig("../Figures/Main/Figure_5/VIP_combined_plot_by_mzRT_sorted_colored_most-specific-class.png", dpi=600, bbox_inches='tight')


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_73083/2328027408.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.15, 0.9, 1])


## Subset metabolomics top VIP table for MMVEC analysis

In [414]:
metaB_feats_to_keep = [655, 1368, 2969, 5930, 23297, 24392, 29059]
metaB_table_subset = metaB_table_subset[metaB_feats_to_keep]

# Create a mapping dictionary: keys are the ID values, values are the corresponding Shortened_Compound_Name
# mapping_dict = VIP_filtered_subset.set_index('ID')['Shortened_Compound_Name'].to_dict()

# Rename the columns in metaB_table_subset using the mapping dictionary
# metaB_table_subset = metaB_table_subset.rename(columns=mapping_dict)


metaB_table_subset


,655,1368,2969,5930,23297,24392,29059
SampleID,,,,,,,
LAMI.RD001.D0.C1,39859.7580,2025607.40,970487.500,567510.060,0.0000,0.00,611709.700
LAMI.RD308.D2.C2,3065.1714,1669351.50,750870.100,442653.660,0.0000,0.00,231525.840
LAMI.RD304.D11.C1,36750.4000,1434033.20,595568.940,328151.120,5375.8525,99470.69,78491.530
LAMI.RD304.D11.C2,0.0000,706197.75,303845.300,152392.720,2691.0256,157503.92,261996.800
LAMI.RD304.D0.C1,3345.8184,778566.10,293386.340,137201.390,3431.0034,0.00,427202.060
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,1250164.60,579058.800,498893.780,0.0000,0.00,122958.260
LAMI.RD308.D9.C3,3076.9688,687527.06,281896.700,262886.470,0.0000,0.00,65659.700
LAMI.RD319.D2.C2,0.0000,538412.94,255542.720,121803.890,0.0000,0.00,73231.560


In [415]:
# Save as csv
metaB_table_subset.to_csv('../Data/metabolomics/Run3_10252024/output/metaB_top-VIPs_final.csv')